# 1. Fourier fitting and explicit validation

Only this notebook generates Fourier coefficients. It saves immutable fits; no classical dynamics are triggered until you explicitly populate `selected_for_dynamics`. Raw fit screening and accepted reduced fits are shown **separately**, so rejected candidates do not introduce misleading NaNs into the saved-fit table.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the checkout so notebooks work without an editable install.
# Supported working directories: repository root or a directory inside it.
_here = Path.cwd().resolve()
ROOT = next(
    (directory for directory in (_here, *_here.parents)
     if (directory / "src" / "bosonic_algo" / "__init__.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the bosonic_algo checkout from the notebook kernel's "
        "working directory. Start Jupyter in the repository root or its "
        "notebooks/ folder (or change the kernel working directory). "
        "For another environment, install from the repository root with "
        "'python -m pip install -e .'."
    )
_src = str(ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from bosonic_algo import generate_fits, read_yaml, selected_fit_classical_validation, list_fits
FIT_ROOT = ROOT / "data" / "fits"
RESULT_ROOT = ROOT / "results"
settings = read_yaml(ROOT / "configs" / "fourier.yaml")

## Generate reduced fits (no time evolution)

`rows` contains **only saved reduced Fourier fits**, with uniform columns. All raw candidates, including rejected ones, are written to `data/fits/raw_fit_summary.csv`.

In [ ]:
rows = generate_fits(settings, root=FIT_ROOT)
saved_df = pd.DataFrame(rows)
print(f"Saved reduced fits: {len(rows)}")
if saved_df.empty:
    print("No raw candidates passed screening; inspect the raw summary below before adjusting raw_v_rmse_max.")
saved_df

## Raw Fourier-fit screening

The raw table contains **every** Fourier candidate and its raw potential/force metrics. A `rejected` status means the raw potential RMSE exceeds `raw_v_rmse_max`; it was not reduced or propagated. This is expected, not missing numerical data.

In [ ]:
raw_df = pd.read_csv(FIT_ROOT / "raw_fit_summary.csv")
print("Raw candidates:", len(raw_df), "| Rejected:", (raw_df["status"] == "rejected").sum())
raw_df

## Inspect candidate fits and select IDs

Update `selected_for_dynamics` in `configs/fourier.yaml` after comparing fit and force metrics. The next cell executes CPU dynamics **only** for those IDs.

In [ ]:
pd.DataFrame(list_fits(FIT_ROOT))

In [ ]:
selected = settings.get("selected_for_dynamics", [])
if not selected:
    print("No classical dynamics requested. Set selected_for_dynamics explicitly.")
else:
    validated = selected_fit_classical_validation(selected, fit_root=FIT_ROOT, result_root=RESULT_ROOT)
    for run in validated:
        print(run["fit_id"], "exact cached:", run["exact"][2], "fitted cached:", run["fitted"][2])